# Проверка обученного RTMDet глазами, не только по mAP

Финальный чекпойнт дал `bbox_mAP=0.964`, `AP50=0.989` на val — но train/val разметка
получена ОДНИМ И ТЕМ ЖЕ SAM3-промптом, так что эта метрика показывает, насколько
RTMDet научился повторять разметку SAM3, а не насколько он прав относительно
реальности. Если у SAM3 была системная ошибка (например, путал кучу мешков на полу
с бельём — см. обсуждение в `experiments/`), RTMDet мог её тоже выучить, и по mAP
это не будет видно, т.к. val-разметка страдает той же ошибкой.

Этот ноутбук берёт кадры со сдвигом относительно `--stride` разметки — то есть кадры,
которых точно не было ни в train, ни в val — и просто рисует предсказания модели на
них, чтобы посмотреть на результат вживую.

Требует CUDA GPU + обученный чекпойнт (`work_dirs/rtmdet_bag/best_coco_bbox_mAP_epoch_*.pth`).
Не запускался в этой сессии — нет GPU и чекпойнта в этом окружении.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
TRAINING_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = TRAINING_DIR.parent

VIDEO_PATH = PROJECT_ROOT / "storage" / "input" / "input.mp4"
CONFIG_PATH = TRAINING_DIR / "configs" / "rtmdet_bag.py"
OUTPUT_DIR = TRAINING_DIR / "outputs" / "predictions_preview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Must match the --stride used in label_with_sam3.py, so the offset below
# lands on frames that were never in train or val.
LABEL_STRIDE = 25
N_SAMPLES = 16
SCORE_THR = 0.3
DEVICE = "cuda:0"

checkpoints = sorted((TRAINING_DIR / "work_dirs" / "rtmdet_bag").glob("best_coco_bbox_mAP_epoch_*.pth"))
assert checkpoints, "no best_coco_bbox_mAP_epoch_*.pth found under work_dirs/rtmdet_bag/ — train first"
CHECKPOINT_PATH = checkpoints[-1]

assert VIDEO_PATH.exists(), f"video not found: {VIDEO_PATH}"
print("checkpoint:", CHECKPOINT_PATH)

## 1. Сэмплирование кадров, НЕ входивших в разметку

`label_with_sam3.py` брал кадры `0, stride, 2*stride, ...`. Здесь берём кадры со
сдвигом `stride // 2` — гарантированно другие индексы, ни разу не показанные модели.

In [ ]:
import cv2
import numpy as np


def sample_unseen_frames(video_path: Path, stride: int, n: int) -> list[tuple[int, np.ndarray]]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    offset = stride // 2
    indices = [offset + i * stride for i in range(n) if offset + i * stride < total_frames]

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame_bgr = cap.read()
        if ok:
            frames.append((idx, frame_bgr))

    cap.release()
    return frames


sampled_frames = sample_unseen_frames(VIDEO_PATH, LABEL_STRIDE, N_SAMPLES)
print(f"sampled {len(sampled_frames)} frames, indices: {[i for i, _ in sampled_frames]}")

## 2. Загрузка модели

In [ ]:
from mmdet.apis import init_detector, inference_detector

model = init_detector(str(CONFIG_PATH), str(CHECKPOINT_PATH), device=DEVICE)

## 3. Предсказания на сэмплированных кадрах

In [ ]:
def predict(frame_bgr: np.ndarray, score_thr: float) -> tuple[np.ndarray, np.ndarray]:
    result = inference_detector(model, frame_bgr)
    instances = result.pred_instances
    keep = instances.scores.cpu().numpy() >= score_thr
    bboxes = instances.bboxes.cpu().numpy()[keep]
    scores = instances.scores.cpu().numpy()[keep]
    return bboxes, scores


predictions = []
for idx, frame_bgr in sampled_frames:
    bboxes, scores = predict(frame_bgr, SCORE_THR)
    predictions.append({"frame_idx": idx, "frame_bgr": frame_bgr, "bboxes": bboxes, "scores": scores})
    print(f"frame {idx:6d} -> {len(scores)} detections (scores: {np.round(scores, 2).tolist()})")

## 4. Визуализация

Смотрим глазами: пропущенные мешки, ложные срабатывания (особенно на куче на полу —
см. заметку в начале ноутбука), качество боксов на маленьких/дальних объектах.

In [ ]:
import matplotlib.pyplot as plt


def draw_predictions(frame_bgr: np.ndarray, bboxes: np.ndarray, scores: np.ndarray) -> np.ndarray:
    vis = frame_bgr.copy()
    for (x1, y1, x2, y2), score in zip(bboxes.astype(int), scores):
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(vis, f"{score:.2f}", (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    return vis


cols = 4
rows = (len(predictions) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3 * rows))
axes = np.array(axes).reshape(-1)

for ax, entry in zip(axes, predictions):
    vis = draw_predictions(entry["frame_bgr"], entry["bboxes"], entry["scores"])
    cv2.imwrite(str(OUTPUT_DIR / f"frame{entry['frame_idx']:06d}.jpg"), vis)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {entry['frame_idx']} ({len(entry['scores'])} det)")
    ax.axis("off")

for ax in axes[len(predictions):]:
    ax.axis("off")

fig.tight_layout()
plt.show()
print(f"saved annotated frames -> {OUTPUT_DIR}")

## Выводы

_Заполнить после просмотра:_

- Совпадает ли субъективное впечатление с mAP=0.964, или модель хуже, чем метрика
  предполагает (из-за circularity — см. верх ноутбука)?
- Ложные срабатывания на куче мешков/белья на полу?
- Пропуски на маленьких/частично перекрытых мешках (AP_small=0.884 — самая слабая
  категория в метриках)?
- Если качество устраивает — этот чекпойнт готов для интеграции в `app/detector.py`
  (следующий шаг, ещё не сделан — см. `../../README.md`, "What's still to decide").